# 🫁 Pneumonia Detection — Training Notebook

Entraîne les trois architectures (**U-Net**, **ResNet**, **Inception**) sur le dataset Chest X-Ray.

## Setup rapide
1. `Runtime > Change runtime type > T4 GPU`
2. Monte ton Google Drive (cellule suivante)
3. Place le dataset dans `MyDrive/pneumonia/data/` avec la structure :
```
data/
  train/  PNEUMONIA/  NORMAL/
  val/    PNEUMONIA/  NORMAL/
  test_for_students/  *.jpeg
```
4. Exécute toutes les cellules dans l'ordre

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# ── Adapte ce chemin si nécessaire ──
PROJECT_ROOT = Path('/content/drive/MyDrive/pneumonia')

TRAIN_DIR   = PROJECT_ROOT / 'data' / 'train'
VAL_DIR     = PROJECT_ROOT / 'data' / 'val'
TEST_DIR    = PROJECT_ROOT / 'data' / 'test_for_students'
WEIGHTS_DIR = PROJECT_ROOT / 'weights'
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

print('Train dir exists :', TRAIN_DIR.exists())
print('Val   dir exists :', VAL_DIR.exists())
print('Test  dir exists :', TEST_DIR.exists())

In [ ]:
!pip install -q scikit-learn

## 1. Définition des modèles

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ── U-Net ──────────────────────────────────────────────────────────────────────

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
    def forward(self, x):
        features = self.conv(x)
        return features, self.pool(features)


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape != skip.shape:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))


class UNet(nn.Module):
    def __init__(self, in_channels=3, base_filters=64, dropout_rate=0.5):
        super().__init__()
        f = base_filters
        self.enc1 = EncoderBlock(in_channels, f)
        self.enc2 = EncoderBlock(f,     f*2)
        self.enc3 = EncoderBlock(f*2,   f*4)
        self.enc4 = EncoderBlock(f*4,   f*8)
        self.bottleneck = DoubleConv(f*8, f*16)
        self.dec4 = DecoderBlock(f*16, f*8)
        self.dec3 = DecoderBlock(f*8,  f*4)
        self.dec2 = DecoderBlock(f*4,  f*2)
        self.dec1 = DecoderBlock(f*2,  f)
        self.pool       = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(f, 1)

    def forward(self, x):
        s1, x = self.enc1(x)
        s2, x = self.enc2(x)
        s3, x = self.enc3(x)
        s4, x = self.enc4(x)
        x = self.bottleneck(x)
        x = self.dec4(x, s4)
        x = self.dec3(x, s3)
        x = self.dec2(x, s2)
        x = self.dec1(x, s1)
        return self.classifier(self.dropout(self.pool(x).flatten(1)))


print('UNet OK — output shape:', UNet()(torch.randn(2, 3, 224, 224)).shape)

In [ ]:
# ── ResNet ─────────────────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    expansion = 1
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.main(x) + self.shortcut(x))


class ResNet(nn.Module):
    def __init__(self, in_channels=3, base_filters=64, dropout_rate=0.5):
        super().__init__()
        self.in_channels = base_filters
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_filters, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(base_filters),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make_layer(base_filters,   2, stride=1)
        self.layer2 = self._make_layer(base_filters*2, 2, stride=2)
        self.layer3 = self._make_layer(base_filters*4, 2, stride=2)
        self.layer4 = self._make_layer(base_filters*8, 2, stride=2)
        self.pool       = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(base_filters*8, 1)
        self._init_weights()

    def _make_layer(self, out_channels, n_blocks, stride):
        layers = [ResidualBlock(self.in_channels, out_channels, stride)]
        self.in_channels = out_channels
        for _ in range(1, n_blocks):
            layers.append(ResidualBlock(self.in_channels, out_channels))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.classifier(self.dropout(self.pool(x).flatten(1)))


print('ResNet OK — output shape:', ResNet()(torch.randn(2, 3, 224, 224)).shape)

In [ ]:
# ── Inception ──────────────────────────────────────────────────────────────────

def conv_bn_relu(in_ch, out_ch, kernel, stride=1, padding=0):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=kernel, stride=stride, padding=padding, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class InceptionBlock(nn.Module):
    def __init__(self, in_channels, b1, b2_reduce, b2_out, b3_reduce, b3_out, b4_out):
        super().__init__()
        self.branch1 = conv_bn_relu(in_channels, b1, kernel=1)
        self.branch2 = nn.Sequential(
            conv_bn_relu(in_channels, b2_reduce, kernel=1),
            conv_bn_relu(b2_reduce,   b2_out,    kernel=3, padding=1),
        )
        self.branch3 = nn.Sequential(
            conv_bn_relu(in_channels, b3_reduce, kernel=1),
            conv_bn_relu(b3_reduce,   b3_out,    kernel=5, padding=2),
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            conv_bn_relu(in_channels, b4_out, kernel=1),
        )
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch3(x), self.branch4(x)], dim=1)


class Inception(nn.Module):
    def __init__(self, in_channels=3, dropout_rate=0.5):
        super().__init__()
        self.stem = nn.Sequential(
            conv_bn_relu(in_channels, 32, kernel=3, stride=2, padding=1),
            conv_bn_relu(32,          64, kernel=3, stride=1, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )
        self.block1 = InceptionBlock(64,  32, 16,  32,  8, 16, 16)   # out: 96
        self.pool1  = nn.MaxPool2d(3, stride=2, padding=1)
        self.block2 = InceptionBlock(96,  64, 32,  64, 16, 32, 32)   # out: 192
        self.pool2  = nn.MaxPool2d(3, stride=2, padding=1)
        self.block3 = InceptionBlock(192, 96, 48,  96, 24, 48, 48)   # out: 288
        self.block4 = InceptionBlock(288, 96, 48,  96, 24, 48, 48)   # out: 288
        self.pool       = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout    = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(288, 1)

    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        x = self.block4(self.block3(x))
        return self.classifier(self.dropout(self.pool(x).flatten(1)))


print('Inception OK — output shape:', Inception()(torch.randn(2, 3, 224, 224)).shape)

## 2. Chargement des données

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 32

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                  std=[0.229, 0.224, 0.225])

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    normalize,
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalize,
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Classes : {train_dataset.classes}  (0=NORMAL, 1=PNEUMONIA selon ordre alpha)')
print(f'Train   : {len(train_dataset)} images')
print(f'Val     : {len(val_dataset)}   images')

## 3. Boucle d'entraînement

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, all_labels, all_probs = 0.0, [], []
    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.float().unsqueeze(1).to(DEVICE)
        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_probs.extend(probs.flatten())
        all_labels.extend(labels.cpu().numpy().flatten())
    avg_loss = total_loss / len(loader.dataset)
    accuracy = float(np.mean((np.array(all_probs) >= 0.5) == np.array(all_labels)))
    auc      = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5
    return avg_loss, accuracy, auc


def evaluate(model, loader, criterion):
    model.eval()
    total_loss, all_labels, all_probs = 0.0, [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)
            logits = model(images)
            loss   = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.extend(probs.flatten())
            all_labels.extend(labels.cpu().numpy().flatten())
    avg_loss = total_loss / len(loader.dataset)
    accuracy = float(np.mean((np.array(all_probs) >= 0.5) == np.array(all_labels)))
    auc      = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5
    return avg_loss, accuracy, auc


def train_model(model, model_name, epochs=20, lr=1e-3, dropout_rate=0.5):
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

    history = dict(train_losses=[], val_losses=[], train_accs=[], val_accs=[], train_aucs=[], val_aucs=[])
    best_val_auc = 0.0

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc, tr_auc = train_one_epoch(model, train_loader, optimizer, criterion)
        vl_loss, vl_acc, vl_auc = evaluate(model, val_loader, criterion)
        scheduler.step(vl_loss)

        for k, v in zip(history, [tr_loss, vl_loss, tr_acc, vl_acc, tr_auc, vl_auc]):
            history[k].append(v)

        if vl_auc > best_val_auc:
            best_val_auc = vl_auc
            save_path = WEIGHTS_DIR / f'{model_name}_best.pt'
            torch.save(model.state_dict(), save_path)
            tag = ' ← best'
        else:
            tag = ''

        print(f'[{model_name}] Epoch {epoch:>2}/{epochs} | '
              f'Train loss {tr_loss:.4f} acc {tr_acc:.4f} auc {tr_auc:.4f} | '
              f'Val   loss {vl_loss:.4f} acc {vl_acc:.4f} auc {vl_auc:.4f}{tag}')

    print(f'\n✅ {model_name} — Best val AUC: {best_val_auc:.4f}')
    return history, best_val_auc

## 4. Entraînement des 3 modèles

Modifie `EPOCHS`, `LR`, `DROPOUT` selon tes besoins.

In [ ]:
EPOCHS       = 20
LR           = 1e-3
DROPOUT_RATE = 0.5

all_histories = {}

In [ ]:
print('=' * 60)
print('U-Net')
print('=' * 60)
unet = UNet(in_channels=3, base_filters=64, dropout_rate=DROPOUT_RATE)
all_histories['U-Net'], _ = train_model(unet, 'U-Net', epochs=EPOCHS, lr=LR)

In [ ]:
print('=' * 60)
print('ResNet')
print('=' * 60)
resnet = ResNet(in_channels=3, base_filters=64, dropout_rate=DROPOUT_RATE)
all_histories['ResNet'], _ = train_model(resnet, 'ResNet', epochs=EPOCHS, lr=LR)

In [ ]:
print('=' * 60)
print('Inception')
print('=' * 60)
inception = Inception(in_channels=3, dropout_rate=DROPOUT_RATE)
all_histories['Inception'], _ = train_model(inception, 'Inception', epochs=EPOCHS, lr=LR)

## 5. Courbes d'entraînement

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def plot_history(history, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    ep = range(1, len(history['train_losses']) + 1)

    for ax, train_key, val_key, title, ylabel in [
        (axes[0], 'train_losses', 'val_losses', 'Loss',     'Loss'),
        (axes[1], 'train_accs',  'val_accs',   'Accuracy', 'Accuracy'),
        (axes[2], 'train_aucs',  'val_aucs',   'ROC-AUC',  'AUC'),
    ]:
        ax.plot(ep, history[train_key], label='Train')
        ax.plot(ep, history[val_key],   label='Val')
        ax.set_title(title); ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.legend()
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

    fig.suptitle(f'{model_name} — Training curves', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()

for name, hist in all_histories.items():
    plot_history(hist, name)

## 6. Comparaison des modèles

In [ ]:
import pandas as pd

rows = []
for name, hist in all_histories.items():
    rows.append({
        'Model':          name,
        'Best Val AUC':   f"{max(hist['val_aucs']):.4f}",
        'Final Val Acc':  f"{hist['val_accs'][-1]:.4f}",
        'Final Val Loss': f"{hist['val_losses'][-1]:.4f}",
    })

pd.DataFrame(rows).set_index('Model')

## 7. Génération du CSV de soumission

Choisis le modèle à utiliser pour la prédiction sur le test set.

In [ ]:
from PIL import Image

# ── Choix du modèle ──────────────────────────────────────────────────────────
SUBMIT_MODEL_NAME = 'ResNet'   # 'U-Net' | 'ResNet' | 'Inception'

MODEL_CLASSES = {'U-Net': UNet, 'ResNet': ResNet, 'Inception': Inception}

# Charge les meilleurs poids
submit_model = MODEL_CLASSES[SUBMIT_MODEL_NAME](in_channels=3).to(DEVICE)
weight_path  = WEIGHTS_DIR / f'{SUBMIT_MODEL_NAME}_best.pt'
submit_model.load_state_dict(torch.load(weight_path, map_location=DEVICE))
submit_model.eval()

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalize,
])

image_paths = sorted(TEST_DIR.glob('*.jpeg')) + \
              sorted(TEST_DIR.glob('*.jpg'))  + \
              sorted(TEST_DIR.glob('*.png'))

print(f'Images de test trouvées : {len(image_paths)}')

records = []
with torch.no_grad():
    for img_path in image_paths:
        img    = Image.open(img_path).convert('RGB')
        tensor = test_transform(img).unsqueeze(0).to(DEVICE)
        prob   = torch.sigmoid(submit_model(tensor)).item()
        records.append({'id': img_path.stem, 'prediction': round(prob, 6)})

df_sub = pd.DataFrame(records)
print(df_sub.head(10))

In [ ]:
from google.colab import files

csv_path = PROJECT_ROOT / f'submission_{SUBMIT_MODEL_NAME}.csv'
df_sub.to_csv(csv_path, index=False)
print(f'Saved to {csv_path}')

# Téléchargement direct dans le navigateur
files.download(str(csv_path))